# 11 — Full-data: huấn luyện 72 run

Chỉ chạy khi benchmark đã hoàn tất và cổng tài nguyên đạt. Checkpoint chỉ hợp lệ sau một lượt đầy đủ qua train; batch không làm giảm số flow.

Chọn kernel **NIDS E-GraphSAGE server** trước khi chạy.

In [ ]:
from pathlib import Path
import json, os, subprocess, sys

ROOT = Path.cwd()
if not (ROOT / 'research/server').is_dir():
    raise RuntimeError('Hãy mở JupyterLab từ thư mục gốc repository')
THREADS = os.environ.get('NIDS_THREADS', '12')
NUM_WORKERS = os.environ.get('NIDS_NUM_WORKERS', '4')
BUDGET_USD = os.environ.get('NIDS_BUDGET_USD', '6')
HOURLY_PRICE_USD = os.environ.get('NIDS_HOURLY_PRICE_USD', '0')

def full_stage(name):
    command = [sys.executable, '-u', 'research/server/run_full_pipeline.py',
               '--stage', name, '--threads', THREADS, '--num-workers', NUM_WORKERS,
               '--budget-usd', BUDGET_USD, '--hourly-price-usd', HOURLY_PRICE_USD]
    print(' '.join(command), flush=True)
    subprocess.run(command, cwd=ROOT, check=True)

print({'root': str(ROOT), 'threads': THREADS, 'num_workers': NUM_WORKERS,
       'budget_usd': BUDGET_USD, 'hourly_price_usd': HOURLY_PRICE_USD,
       'python': sys.executable,
       'pipeline': 'full-data benchmark-gated'})


In [ ]:
estimate = json.loads((ROOT / 'research/results/full_benchmark_estimate.json').read_text())
if estimate.get('safe_to_launch_72') is not True:
    raise RuntimeError('Cổng benchmark chưa đạt; không được chạy 72 run')
print(json.dumps(estimate['step_budget_estimate'], indent=2))


## Khởi chạy có resume

Run đã hoàn thành được giữ nguyên khi notebook chạy lại.

In [ ]:
full_stage('train')
